In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

def mostrar_imagem_aleatoria(pasta, label):
    arquivos = os.listdir(pasta)
    escolhido = random.choice(arquivos)

    img = Image.open(os.path.join(pasta, escolhido))

    plt.imshow(img, cmap="gray")
    plt.title(label)
    plt.axis("off")
    plt.show()



mostrar_imagem_aleatoria("data/train/cat", "Gato")


mostrar_imagem_aleatoria("data/train/dog", "Cachorro")

In [ ]:
def preprocessar_imagem(caminho):   #conversão pra IA burra que nao entende nada
    img = Image.open(caminho)

    img = img.resize((64, 64))
    img = img.convert("L")

    img = np.array(img).astype(np.float32) / 255.0
    img = img.flatten()

    return img

In [ ]:
def carregar_dados():           #classificacao cao/gato
    X = []
    y = []

    for img_name in os.listdir("data/train/cat"):
        path = "data/train/cat/" + img_name
        X.append(preprocessar_imagem(path))
        y.append(0)

    for img_name in os.listdir("data/train/dog"):
        path = "data/train/dog/" + img_name
        X.append(preprocessar_imagem(path))
        y.append(1)

    return np.array(X), np.array(y)


X, y = carregar_dados()

In [ ]:
X = X.astype(np.float32)
y = y.astype(np.float32)


X = (X - X.mean()) / (X.std() + 1e-8)

indices = np.random.permutation(len(X))              #aleatoriedade 
X = X[indices]
y = y[indices]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
                                       # ativação de valores positivos.
def relu(x):
    return np.maximum(0, x)

In [ ]:
entrada = 4096
oculta = 128
saida = 1

np.random.seed(42)


W1 = np.random.randn(entrada, oculta) * np.sqrt(2 / entrada)
b1 = np.zeros((1, oculta))

W2 = np.random.randn(oculta, saida) * np.sqrt(2 / oculta)
b2 = np.zeros((1, saida))

In [ ]:
lr = 0.01   
epochs = 50

In [ ]:
def treinar(X, y):
    global W1, b1, W2, b2

    m = X.shape[0]

    for epoch in range(epochs):

            
        Z1 = X @ W1 + b1
        A1 = relu(Z1)

        Z2 = A1 @ W2 + b2
        A2 = sigmoid(Z2)

        A2 = np.clip(A2, 1e-8, 1 - 1e-8)

        
        loss = -np.mean(
            y.reshape(-1,1) * np.log(A2) +
            (1 - y.reshape(-1,1)) * np.log(1 - A2)
        )

       
        preds = (A2 > 0.5).astype(int)
        acc = np.mean(preds == y.reshape(-1,1))

        print(f"Epoch {epoch+1:02d} - Loss: {loss:.4f} - Acc: {acc:.4f}")

       
        dZ2 = A2 - y.reshape(-1,1)

        dW2 = (A1.T @ dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        dA1 = dZ2 @ W2.T
        dZ1 = dA1 * (Z1 > 0)

        dW1 = (X.T @ dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

In [ ]:
treinar(X, y)